In [30]:
import torch
import torch.nn as nn
import math

pairs = [
    ("我愛你", "i love you"),
    ("我喜歡貓", "i like cats"),
    ("我喜歡狗", "i like dogs"),
    ("他是學生", "he is a student"),
    ("她是老師", "she is a teacher"),
    ("今天天氣好", "the weather is good today"),
    ("我正在學習", "i am learning"),
    ("我學習機器學習", "i study machine learning"),
    ("這是一本書", "this is a book"),
    ("你喜歡音樂", "you like music"),
]

In [31]:
SRC_PAD = "<pad>" # 中文端 padding token，用來把不同長度的中文句子補成一樣長。
SRC_UNK = "<unk>" # 中文端 UNK token，是 unknown 的意思，用來表示「字彙表裡沒有的字」。

TGT_PAD = "<pad>" # 英文端 padding token，用來把不同長度的中文句子補成一樣長。
TGT_UNK = "<unk>" # 英文端 UNK token，是 unknown 的意思，用來表示「字彙表裡沒有的字」。
TGT_SOS = "<sos>" # Start token
TGT_EOS = "<eos>" # End token

src_chars = sorted(list(set("".join(src for src, tgt in pairs))))

src_itos = [SRC_PAD, SRC_UNK] + src_chars
src_stoi = {ch: i for i, ch in enumerate(src_itos)}

src_pad_id = src_stoi[SRC_PAD]
src_unk_id = src_stoi[SRC_UNK]

print(src_stoi)


{'<pad>': 0, '<unk>': 1, '一': 2, '今': 3, '他': 4, '你': 5, '喜': 6, '器': 7, '在': 8, '天': 9, '她': 10, '好': 11, '學': 12, '師': 13, '愛': 14, '我': 15, '是': 16, '書': 17, '本': 18, '樂': 19, '機': 20, '歡': 21, '正': 22, '氣': 23, '狗': 24, '生': 25, '習': 26, '老': 27, '貓': 28, '這': 29, '音': 30}


In [32]:
tgt_chars = sorted(list(set("".join(tgt for src, tgt in pairs))))

tgt_itos = [TGT_PAD, TGT_UNK, TGT_SOS, TGT_EOS] + tgt_chars
tgt_stoi = {ch: i for i, ch in enumerate(tgt_itos)}

tgt_pad_id = tgt_stoi[TGT_PAD]
tgt_unk_id = tgt_stoi[TGT_UNK]
tgt_sos_id = tgt_stoi[TGT_SOS]
tgt_eos_id = tgt_stoi[TGT_EOS]

print(tgt_stoi)

{'<pad>': 0, '<unk>': 1, '<sos>': 2, '<eos>': 3, ' ': 4, 'a': 5, 'b': 6, 'c': 7, 'd': 8, 'e': 9, 'g': 10, 'h': 11, 'i': 12, 'k': 13, 'l': 14, 'm': 15, 'n': 16, 'o': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'y': 24}


In [33]:
def encode_src(text): # 將中文字轉成 token id
    return [src_stoi.get(ch, src_unk_id) for ch in text]
def encode_tgt(text): # 將英文字轉成 token id
    return [tgt_sos_id] + [tgt_stoi.get(ch, tgt_unk_id) for ch in text] + [tgt_eos_id]
print(encode_src("我愛你"))
print(encode_tgt("i love you"))

[15, 14, 5]
[2, 12, 4, 14, 17, 22, 9, 4, 24, 17, 21, 3]


In [34]:
def pad_sequences(sequences, pad_id): # 把 batch 補成相同長度
    max_len = max(len(seq) for seq in sequences)
    result = []

    for seq in sequences:
        padded = seq + [pad_id] * (max_len - len(seq))
        result.append(padded)

    return torch.tensor(result, dtype=torch.long)

In [35]:
src_sequences = [encode_src(src) for src, tgt in pairs]
tgt_sequences = [encode_tgt(tgt) for src, tgt in pairs]

print(src_sequences)
src_batch = pad_sequences(src_sequences, src_pad_id)
tgt_batch = pad_sequences(tgt_sequences, tgt_pad_id)

print(src_batch.shape)
print(tgt_batch.shape)

tgt_input = tgt_batch[:, :-1]
tgt_output = tgt_batch[:, 1:]

print(tgt_input.shape)
print(tgt_output.shape)

[[15, 14, 5], [15, 6, 21, 28], [15, 6, 21, 24], [4, 16, 12, 25], [10, 16, 27, 13], [3, 9, 9, 23, 11], [15, 22, 8, 12, 26], [15, 12, 26, 20, 7, 12, 26], [29, 16, 2, 18, 17], [5, 6, 21, 30, 19]]
torch.Size([10, 7])
torch.Size([10, 27])
torch.Size([10, 26])
torch.Size([10, 26])


In [36]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [37]:
class TransformerTranslator(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=64,
        nhead=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        dim_feedforward=128,
        dropout=0.1
    ):
        super().__init__()

        self.d_model = d_model

        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

        self.pos_encoder = PositionalEncoding(d_model)
        self.pos_decoder = PositionalEncoding(d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def forward(
        self,
        src,
        tgt,
        src_key_padding_mask=None,
        tgt_key_padding_mask=None,
        memory_key_padding_mask=None,
        tgt_mask=None
    ):
        src_emb = self.src_embedding(src) * math.sqrt(self.d_model)
        tgt_emb = self.tgt_embedding(tgt) * math.sqrt(self.d_model)

        src_emb = self.pos_encoder(src_emb)
        tgt_emb = self.pos_decoder(tgt_emb)

        output = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )

        logits = self.fc_out(output)
        return logits

In [38]:
def generate_square_subsequent_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    mask = mask.masked_fill(mask == 1, float("-inf"))
    return mask
def create_padding_mask(seq, pad_id):
    return seq == pad_id

In [39]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TransformerTranslator(
    src_vocab_size=len(src_itos),
    tgt_vocab_size=len(tgt_itos),
    d_model=64,
    nhead=4,
    num_encoder_layers=2,
    num_decoder_layers=2
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

loss_fn = nn.CrossEntropyLoss(ignore_index=tgt_pad_id)

In [40]:
src_batch = src_batch.to(device)
tgt_input = tgt_input.to(device)
tgt_output = tgt_output.to(device)

In [41]:
for epoch in range(1000):
    model.train()

    src_batch_dev = src_batch.to(device)
    tgt_input_dev = tgt_input.to(device)
    tgt_output_dev = tgt_output.to(device)

    tgt_mask = generate_square_subsequent_mask(tgt_input.size(1)).to(device)

    src_padding_mask = create_padding_mask(src_batch, src_pad_id).to(device)
    tgt_padding_mask = create_padding_mask(tgt_input, tgt_pad_id).to(device)

    logits = model(
        src_batch_dev,
        tgt_input_dev,
        src_key_padding_mask=src_padding_mask,
        tgt_key_padding_mask=tgt_padding_mask,
        memory_key_padding_mask=src_padding_mask,
        tgt_mask=tgt_mask
    )

    loss = loss_fn(
        logits.reshape(-1, len(tgt_itos)),
        tgt_output_dev.reshape(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"epoch {epoch}, loss = {loss.item():.4f}")

epoch 0, loss = 3.3747


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


epoch 100, loss = 0.2377
epoch 200, loss = 0.0686
epoch 300, loss = 0.0275
epoch 400, loss = 0.0266
epoch 500, loss = 0.0240
epoch 600, loss = 0.0157
epoch 700, loss = 0.0119
epoch 800, loss = 0.0111
epoch 900, loss = 0.0150


In [42]:
def translate(model, src_text, max_len=50):
    model.eval()

    src_ids = encode_src(src_text)
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)

    src_padding_mask = create_padding_mask(src, src_pad_id).to(device)

    tgt_ids = [tgt_sos_id]

    for _ in range(max_len):
        tgt = torch.tensor(tgt_ids, dtype=torch.long).unsqueeze(0).to(device)

        tgt_mask = generate_square_subsequent_mask(tgt.size(1)).to(device)

        logits = model(
            src,
            tgt,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=None,
            memory_key_padding_mask=src_padding_mask,
            tgt_mask=tgt_mask
        )

        next_logits = logits[:, -1, :]
        next_id = torch.argmax(next_logits, dim=-1).item()

        if next_id == tgt_eos_id:
            break

        tgt_ids.append(next_id)

    result = "".join(tgt_itos[i] for i in tgt_ids[1:])
    return result

print(translate(model, "我愛你"))
print(translate(model, "我喜歡貓"))
print(translate(model, "他是學生"))
print(translate(model, "這是一本書"))
print(translate(model, "我喜歡機器學習"))

i love you
i like cats
he is a student
this is a book
i study machine learning
